# Week 15 - PDF Parsing

When designing the course I've asked myself "What has been personally useful to me as a programmer?"

One such topic which came to mind is PDF Parsing.

There are many different open-source community built libraries for PDF parsing.

I recently made a program to parse data from applicants to our Master's Program. To do so I tried a few libraries, and found the one that worked for me was called "pdfminer". The most recent release can be installed through the library name "pdfminer-six".

So let's install it and get started. Along the way we'll also touch on other new topics and coding techniques.

In [13]:
%pip install pdfminer-six

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### A Very Simple Example

Parsing the text from an admissions questionnaire.

In [ ]:
from pdfminer.high_level import extract_text

In [3]:
file = './questionnaire.pdf'

text = extract_text(file)

In [4]:
text

'DIGITAL CULTURAL HER ITAGE  STUDIES  \n\nQuestionnaire for the Admission Procedure \nM. A. Digital Cultural Heritage\n\nI Personal Data \n\nAddress / Surname: \n\nFirst name(s): \n\nDate of Birth: \n\nPlace/Country of Birth: \n\nNationality: \n\nCurrent Home Address: \n\nE-mail Address: \n\nTelephone: \n\nII Details on Previous Studies \n\nPlease  provide  us  with  the  full  details  on  all  previous  university  degrees,  that  you  have \nearned so far, or that you are going to complete before the start date of the M. A. program \nDigital Cultural Heritage in October of the current year. In case of more than one program, \nuse the second and third column respectively. \n\n1 \n\n2 \n\n3 \n\nDegree: \n(B. A., B. Sc., M.A., \nM. Sc. etc.)\nFull Name of \nProgram: \n(and minors if \napplicable) \n\nFull name of \nUniversity: \n\nYear of \ngraduation \n\n\x0cLUDWIG-MAXIMILIANS-UNI VERSITÄT MÜNCHEN  \n\nSEITE 2 VON  2 \n\nIII Agreement on Notification by E-mail \n\nIf  the  admission  

But wait, the filled-in fields are not included. How do we get those?

The data structure for a PDF is quite complicated. It is stored in a complex dictionary which includes many data types (and even some nested dictonaries!).

The filled in data is under the "AcroForm" key, and within this the individual entries are stored under the "fields" key.

In [ ]:
from pdfminer.pdfparser import PDFParser
from pdfminer.pdfdocument import PDFDocument
from pdfminer.psparser import PSLiteral
from pdfminer.pdftypes import resolve1, PDFObjRef

file = './questionnaire.pdf'

data = {}

with open(file, 'rb') as fp:
    parser = PDFParser(fp)
    doc = PDFDocument(parser)

    # The "resolve1" function changes the pdf datatypes to a normal dictionary python can handle
    fields = resolve1(doc.catalog['AcroForm'])['Fields']
    for f in fields:
        # For each filled-in field in the form the following two lines return the name of the field (name), and the entered text (value)
        field = resolve1(f)
        name, value = field.get('T'), field.get('V')

        # Unfortunately there can be instances the name and value are not normal strings
        # In this case the following 3 "if statements" are necessary to deal with that
        if isinstance(name, PSLiteral):
            name = name.name
        if isinstance(value, PDFObjRef):
            value = resolve1(value)
        if isinstance(value, PSLiteral):
            value = value.name
        
        # The field values are in binary. We need to change that to standard "utf-8" encoding to view them
        if value is not None and not isinstance(value,(str,dict)):
            value = value.decode('utf-8')
            
        # And now we can save the decoded information in a key-value pair in our data dictionary
        data[name.decode('utf-8')] = value

In [9]:
data

{'Surname': 'Eames',
 'First name(s)': 'Evan',
 'Date of Birth': '22/07/1990',
 'Place of Birth': 'Canada',
 'Nationality': 'Canadian',
 'Current Home Address': 'Holzstr. 37, 80469, Munich, DE',
 'E-mail Address': 'evan.eames@lmu.de',
 'Telephone': '01783717245',
 '1st Degree': 'BSc',
 '2nd Degree': 'MSc',
 '3rd Degree': 'PhD',
 '2 Full Name of Program': 'Astrophysics',
 '1 Full Name of Program': 'Honours Physics',
 '3 Full Name of Program': 'Astrophysics',
 '1 University': 'McGill University',
 '2 University': 'University of Manchester',
 '3 University': 'Paris Sciences et Lettres',
 '1 Year of graduation': '2014',
 '2 Year of graduation': '2015',
 '3 Year of graduation': '2018',
 'E-mail-Notification': 'Yes',
 'Place name': 'Munich',
 'Date of signature': '10/05/2026',
 'Signature': None,
 'Form of address': 'Mr.'}

In [ ]:
# Just for fun, we can look at the rest of the PDF data structure.

doc.catalog

{'AcroForm': <PDFObjRef:158>,
 'Lang': b'de-DE',
 'MarkInfo': {'Marked': True},
 'Metadata': <PDFObjRef:14>,
 'Pages': <PDFObjRef:128>,
 'StructTreeRoot': <PDFObjRef:34>,
 'Type': /'Catalog',
 'ViewerPreferences': <PDFObjRef:159>}

# Some more stuff

### Checking what's in a directory

In [11]:
import os

directory_path = '../../../Admissions_2026'

dir_names = []

with os.scandir(directory_path) as entries:
    for entry in entries:
        if entry.is_dir():
            dir_names.append(entry.name)

print(dir_names)


['001_Yuhan_XIA', '002_Elizabeth_LONGO', '003_Sute_IWAR', '004_Crescenzo_DIGIANNANTONIO', '005_Liu_ZIJIA', '006_Matthew_CURRIE', '007_Ezinne_Favour_NDIONYEMA', '008_Mohammad_Shamiur_RAHMAN', '009_Dennis_UMEAKUBILO', '010_Jacinta_ATAYDE', '011_Yanrong_ZHU', '012_Muhammed_Mansoor_GUJJAR', '013_Saba_TARIQ', '014_Zhen_QIAN', '015_Manlin_MA']


### Checking which files are in a directory

In [ ]:
for dir in dir_names:
    print(dir)
    applicant = initializeApplicant(dir)
    with os.scandir(directory_path + '/' + dir) as entries:
        for entry in entries:
            if entry.is_file():
                if 'questionnaire' in entry.name.lower():
                    applicant.questionnaire = True
                    questionnaire_file = entry.name
                    applicant_dir = directory_path + '/' + dir + '/' + questionnaire_file
                    extractApplicantInfo(applicant_dir, applicant)

### Try-Except Block

In [18]:
# This breaks stuff

1/0

print("Continuing...")

ZeroDivisionError: division by zero

In [19]:
# This doesn't

try:
    1/0
except:
    print("You can't divide by zero!")

print("Continuing...")

You can't divide by zero!
Continuing...


# Project

### System to Process Applications